# 🔬 Feature Engineering Example

This notebook demonstrates how to fit feature transformers (Scaler, OneHot) on training data and apply them to test data.

## Scenario
1. **Fit**: Learn mean/std from `train_data.csv` for scaling, and categories for OneHot encoding.
2. **Transform**: Apply the learned parameters to `test_data.csv`.

## 🔧 Setup

Install `mlprep-rust` package from PyPI.

In [ ]:
!pip install mlprep-rust -q

## 📁 Generate Train/Test Data

Generate training and test datasets with numeric and categorical features.

In [ ]:
import pandas as pd
import numpy as np

def generate_train_test():
    np.random.seed(42)
    n_train = 100
    n_test = 20
    
    cities = ['Tokyo', 'Osaka', 'Nagoya', 'Fukuoka']
    
    # Train Data
    train = pd.DataFrame({
        'id': range(1, n_train + 1),
        'age': np.random.randint(20, 60, size=n_train),
        'income': np.random.randint(300, 1000, size=n_train) * 10000,
        'city': np.random.choice(cities, size=n_train)
    })
    
    # Test Data
    test = pd.DataFrame({
        'id': range(n_train + 1, n_train + n_test + 1),
        'age': np.random.randint(20, 60, size=n_test),
        'income': np.random.randint(300, 1000, size=n_test) * 10000,
        'city': np.random.choice(cities, size=n_test)
    })
    
    train.to_csv('train_data.csv', index=False)
    test.to_csv('test_data.csv', index=False)
    print("Generated train_data.csv and test_data.csv")
    return train, test

train_df, test_df = generate_train_test()

print("\n📊 Train Data (first 5 rows):")
print(train_df.head())

print("\n📊 Test Data (first 5 rows):")
print(test_df.head())

print("\n📈 Train Statistics:")
print(train_df.describe())

## 📝 Create Pipeline Configurations

Create separate pipeline files for training and test data.

In [ ]:
# Train pipeline
pipeline_train_yaml = """
name: feature_eng_train
inputs:
  - path: train_data.csv
    format: csv

steps:
  - type: features
    config:
      features:
        - column: age
          transform: standard_scale
        - column: income
          transform: standard_scale
        - column: city
          transform: one_hot_encode

outputs:
  - path: train_features.parquet
    format: parquet
"""

# Test pipeline
pipeline_test_yaml = """
name: feature_eng_test
inputs:
  - path: test_data.csv
    format: csv

steps:
  - type: features
    config:
      features:
        - column: age
          transform: standard_scale
        - column: income
          transform: standard_scale
        - column: city
          transform: one_hot_encode

outputs:
  - path: test_features.parquet
    format: parquet
"""

with open('pipeline_train.yaml', 'w') as f:
    f.write(pipeline_train_yaml.strip())

with open('pipeline_test.yaml', 'w') as f:
    f.write(pipeline_test_yaml.strip())

print("Created pipeline_train.yaml and pipeline_test.yaml")
print("\n📄 Train Pipeline:")
print(pipeline_train_yaml)

## 🚀 Run Train Pipeline

Fit and transform training data.

In [ ]:
!mlprep run pipeline_train.yaml

## 🚀 Run Test Pipeline

Transform test data using the same feature settings.

In [ ]:
!mlprep run pipeline_test.yaml

## ✅ Verify Output

Compare train and test features.

In [ ]:
import pandas as pd
import os

if os.path.exists('train_features.parquet'):
    train_features = pd.read_parquet('train_features.parquet')
    print(f"✅ Train Features: {train_features.shape}")
    print("\nColumns:", train_features.columns.tolist())
    print("\nFirst 5 rows:")
    print(train_features.head())
else:
    print("❌ train_features.parquet not found")

print("\n" + "="*50)

if os.path.exists('test_features.parquet'):
    test_features = pd.read_parquet('test_features.parquet')
    print(f"\n✅ Test Features: {test_features.shape}")
    print("\nColumns:", test_features.columns.tolist())
    print("\nFirst 5 rows:")
    print(test_features.head())
else:
    print("\n❌ test_features.parquet not found")

## 📊 Summary

Analyze the transformed features.

In [ ]:
if os.path.exists('train_features.parquet') and os.path.exists('test_features.parquet'):
    train_features = pd.read_parquet('train_features.parquet')
    test_features = pd.read_parquet('test_features.parquet')
    
    print("📊 Feature Engineering Summary")
    print("="*50)
    print(f"Original train columns: {train_df.columns.tolist()}")
    print(f"Transformed train columns: {train_features.columns.tolist()}")
    print(f"\nOriginal train shape: {train_df.shape}")
    print(f"Transformed train shape: {train_features.shape}")
    
    print("\n📈 Scaled Feature Statistics (Train):")
    if 'age' in train_features.columns:
        print(f"  age - mean: {train_features['age'].mean():.4f}, std: {train_features['age'].std():.4f}")
    if 'income' in train_features.columns:
        print(f"  income - mean: {train_features['income'].mean():.4f}, std: {train_features['income'].std():.4f}")
    
    # Check one-hot encoded columns
    onehot_cols = [c for c in train_features.columns if c.startswith('city_')]
    if onehot_cols:
        print(f"\n🏙️ One-Hot Encoded City Columns: {onehot_cols}")